# Dunnhumby seed48: N에 따른 가치 대응행렬 M2 개발 실험

기존 M1 결과를 재사용하고, 기존 M2와 새 M2만 한 런타임에서 순차 학습합니다.
- 런타임 → 런타임 유형 변경 → **L4 GPU**를 선택하세요.
- seed48, 300 epoch 고정. 25epoch마다 전체 평가, 매epoch checkpoint·optimizer·RNG 저장.
- 100/200은 학습곡선 참고용, 판정은 300. 얼리스토핑·최고epoch 선택 없음.
- 개발684–690일만 사용합니다. 코드의 내부 split 이름 test는 역사적 개발구간이며 보호 test698–704·holdout은 열지 않습니다.
- 기존: q_C·g_N(q_N)·b(q_V). 새: q_C·[T0+(2q_N−1)TN]·b(q_V). 상품: b(상품 구매금액 백분위).
- 두 arm 모두 ρ=.25, RBF폭=.25, ID64, 2층, ID L2=.001, lr=.0005, batch8192, binary graph, uniform K1, plain BPR.
- 새 행렬은 T0=I, TN=0으로 시작합니다. N 직접효과는 처음 0이지만 첫 BPR부터 TN gradient가 흐릅니다. 행렬 별도 규제·손실 추가는 없습니다.
- 2개 스칼라 게이트를 18개 행렬 파라미터로 바꾸므로 용량 차이가 있습니다. 성능 상승만으로 CLV 귀속을 주장하지 않습니다.
- 2026-10-05 사용자 지시에 따른 판정 용도 변경: M2 단독은 진단용입니다. 아래 기존 candidate 값은 단독 기준 충족 여부이며, false여도 결합 실험에서 탈락시키지 않습니다.
- 기존 단독 기준은 보존: 전체 Recall/NDCG@10·20·50 각각 M1의99% 이상, 전체 가중 적중값@10·가중 NDCG@10 모두 M1 초과. 기존 M2 대비 결과도 별도 보고합니다.
- 최종 보완성은 후속 M2+M3+M4 대 M3+M4의 동일조건 비교로 판단합니다. M1만 이기고 M3+M4를 개선하지 못하면 M2의 추가효과로 인정하지 않습니다. 이 노트북에는 결합 학습이 없습니다.
- seed48은 이미 노출된 개발시드입니다. 이번 결과는 탐색이며 유의성·일반화·M5 개선을 주장하지 않습니다.
- 노트북은 로컬에서 문법·소형 CPU 학습/재개 테스트만 완료했습니다. 실제 GPU·Drive 데이터 실행은 이 노트북에서 진행합니다.


## 1. Drive 연결과 고정 코드 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, torch
SOURCE_COMMIT = 'e4e51aec6e2447acafd000f4b7905f5d9d34aaf3'
REPO = Path('/content/clv-m2-conditional-basis-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
assert torch.cuda.is_available(), 'GPU 런타임을 연결하세요. L4 권장.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


## 2. 데이터·M1 기준 확인 — 아직 학습하지 않음
M1 기준파일이 없거나 입력·설정이 다르면 중단합니다. M1을 자동 재학습하지 않습니다.

In [ ]:
import clv_m2_nv_conditional_basis_screen as screen
cfg = screen.configure()
assert Path(cfg.baseline_json).is_file(), f'기존 seed48 M1 결과가 필요합니다: {cfg.baseline_json}'
cfg, prepared = screen.prepare(cfg)


## 3. 기존 M2 → 새 M2 순차 학습
M1 재학습·M5 추가학습 없음. 완료 arm은 재사용하며 중단 arm은 마지막 완료 epoch에서 재개합니다. 학습 로그는 25epoch 평가 시 출력되며 진행상황은 out_dir/progress 아래에도 저장됩니다.

In [ ]:
result = screen.run(cfg, prepared)
print('아래 candidate는 기존 M2 단독 기준 충족 여부입니다. 미달이어도 결합 후보 탈락 아님. M5 보완성은 아직 미평가.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


## 4. 전체 사용자 주요지표 비교 — 모든 세그먼트·노출지표는 CSV에 포함

In [ ]:
metrics = [f'{m}@{k}' for k in (10,20,50) for m in ('recall','ndcg','price_purchase_amount_weighted_hit','vndcg')]
absolute = result['absolute']
for epoch in (100, 200, 300):
    print(f'\n{epoch} epoch 절대지표')
    print(absolute[absolute.epoch.eq(epoch)].set_index('model_id')[metrics].T.to_string())
comparison = result['comparison']
view = comparison[comparison.epoch.eq(300) & comparison.metric.isin(metrics)].copy()
view['relative_change_pct'] = 100 * (view['ratio'] - 1)
print('\n300 epoch 상대 변화율(%)')
print(view.pivot(index='metric', columns=['model_id','reference'], values='relative_change_pct').to_string())
print('\nN 작동 진단')
diag = result['diagnostics']
print(diag[diag.model_id.eq(screen.NEW)].to_string(index=False))
print(json.dumps(result['paths'], ensure_ascii=False, indent=2))


## 5. 결과 ZIP 다운로드
absolute·comparison·diagnostics·result.json을 묶습니다. 큰 checkpoint는 Drive에 남겨 둡니다.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/m2_nv_conditional_basis_seed48_results', 'zip', str(Path(cfg.out_dir) / 'reports'))
files.download(archive)
